[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/sandbox/pharmaconet_smoke_test.ipynb)

# PharmacoNet smoke test

**Sandbox notebook - not workshop material.**

The only question this notebook answers is: *does PharmacoNet install and run in Colab today,
on the blue group's real CpABC1-silymarin complex?*

PharmacoNet turns a protein pocket into a pharmacophore: a 3D map of the chemical features a
molecule would need in order to bind there
([Seo and Kim, Chemical Science 2024](https://doi.org/10.1039/D4SC04854G),
[repository](https://github.com/SeonghwanSeo/PharmacoNet), MIT licence).

It **detects** features that the pocket already implies. It does not invent them, and it never
sees the reference ligand except to pick the centre of the box it looks at. That is what makes
section 8 a real test: if the features it finds land on silybin anyway, they are telling us
something about the pocket.

## Before you run this

Set the **runtime version to 2026.07** (*Runtime > Change runtime type*). This notebook carries
that pin in its metadata, but check it anyway.

PharmacoNet declares `requires-python = ">=3.10,<3.13"`, and pip enforces that even for a
`git+` install. Runtime 2026.07 is Python 3.12.13 and is supported until July 2027. If it has
been retired by the time you read this, 2026.04, 2026.01 and 2025.10 are also Python 3.12.

**No GPU is needed.** The authors quote about a minute on one CPU, so asking for a T4 only makes
you queue for hardware you will not use.

> **Note:** this notebook runs on the blue group's **unpublished** CpABC1 model, which is not in
> this public repository. Section 5 asks you to upload two files by hand from your local clone.

## What you will do

- Record what Python and Colab image this runtime actually has
- Install PharmacoNet from GitHub and fetch its pretrained weights
- Run it on the authors' own example first, so a failure is diagnosable
- Run it on CpABC1 with silybin and read the pharmacophore out
- Check whether the detected features land on silybin's known contacts

## 1. Inspect the runtime

Record what we have before installing anything. This cell also sets up the pass/fail ledger that
every later section writes into, and that section 11 prints as one block you can paste back.

The Python version is the thing to watch. If it is 3.13, nothing below will work, and the error
will look like a PharmacoNet bug when it is really a runtime setting.

In [ ]:
import os, platform, sys

RESULT = {}

def check(step, ok, detail=""):
    """Record a pass or fail for one step, print it, and return it."""
    RESULT[step] = ("PASS" if ok else "FAIL", detail)
    print(f"[{RESULT[step][0]}] {step}" + (f" - {detail}" if detail else ""))
    return ok

print("Colab image:", os.environ.get("COLAB_RELEASE_TAG", "not on Colab"))
print("Python", platform.python_version())
ok = (3, 10) <= sys.version_info[:2] < (3, 13)
check("1. python >=3.10,<3.13", ok, platform.python_version())
if not ok:
    print("\nFIX: Runtime > Change runtime type > runtime version 2026.07,"
          "\nthen Runtime > Disconnect and delete runtime, then run this cell again.")

If that said FAIL, stop here and fix the runtime version. Everything below will fail in a
way that looks like a PharmacoNet problem but is not.

## 2. Install PharmacoNet

PharmacoNet has no PyPI release, so it installs straight from GitHub. We pin a commit rather than
following `main`, because the only tag (`v2.2.0-alpha2`) is named "alpha" but points at the commit
whose message is "Finish v2.2.0" - not something to rely on staying put.

What comes with it: `torch`, `numpy`, `numba`, `omegaconf`, `molvoxel`, `gdown`, `rdkit`,
`openbabel-wheel`, `biopython`. **No DGL, no conda, no kernel restart.** That is the whole reason
this notebook is short and the PharmacoForge one is not.

> **Note:** pip may downgrade `numpy` to satisfy `numba`. If Colab then tells you to restart the
> session, do it and carry on from the next cell - nothing above needs re-running.

In [ ]:
COMMIT = "0f2feec4e398b618fa04ce6eca8f86d2855160f4"  # main, 2025-07-15, "Finish v2.2.0"
!pip install -q "pharmaconet @ git+https://github.com/SeonghwanSeo/PharmacoNet.git@{COMMIT}"
!pip install -q py3Dmol

Check that the imports which usually break come up together.

Also check for the `obabel` command-line program. PharmacoNet's pocket extraction shells out to it
with `os.system(...)` and throws away both the error message and the return code, so if it is
missing you get no warning at all - just a slightly different pocket.

In [ ]:
import shutil
import importlib.metadata as meta

try:
    import numba, numpy, pmnet, rdkit, torch
    from pmnet.module import PharmacoNet
    versions = f"pmnet {meta.version('pharmaconet')} | torch {torch.__version__} | numpy {numpy.__version__}"
    check("2. imports", True, versions)
except Exception as error:
    check("2. imports", False, f"{type(error).__name__}: {error}")

check("2b. obabel on PATH", shutil.which("obabel") is not None, shutil.which("obabel") or "missing")

## 3. Fetch the model weights

PharmacoNet downloads its weights lazily the first time you build the model. We do it here,
explicitly, so that a download failure shows up as its own FAIL rather than as a confusing
traceback inside the first run.

> **Note:** this is the most fragile step in the notebook. The authors host the weights on a
> single Google Drive link with no mirror, and Drive rate-limits public files - "Too many users
> have viewed or downloaded this file recently"
> ([issue #12](https://github.com/SeonghwanSeo/PharmacoNet/issues/12)). As of 2026-09-24 that link
> is quota-exceeded. The cell tries a mirror on this workshop's own releases first; if that does
> not exist yet it falls back to the authors' link.

In [ ]:
from pathlib import Path
import urllib.error, urllib.request

MIRROR = ("https://github.com/ersilia-os/ub-cedd-projects-workshop/"
          "releases/download/pmnet-weights-v2.2.0/pmnet.tar")
DRIVE = "https://drive.google.com/uc?id=1gzjdM7bD3jPm23LBcDXtkSk18nETL04p"
WEIGHTS, source = Path("pmnet.tar"), "not downloaded"

if not WEIGHTS.exists():
    try:
        urllib.request.urlretrieve(MIRROR, WEIGHTS)
        source = "workshop mirror"
    except urllib.error.HTTPError:
        import gdown
        gdown.download(DRIVE, str(WEIGHTS), quiet=False)
        source = "authors' Google Drive"

size = WEIGHTS.stat().st_size / 1e6 if WEIGHTS.exists() else 0
check("3. weights", size > 1, f"{size:.0f} MB from {source}")

## 4. Run the authors' own example first

Before touching CpABC1, run the case the authors ship and document: **6OIM**, the KRAS G12C
oncoprotein with the covalent inhibitor adagrasib. Both files come straight out of their
repository, so there is nothing for us to prepare wrongly.

The point is diagnosis. If this works and CpABC1 does not, the problem is our input or our target.
If this fails too, the problem is the install or the weights, and nothing about CpABC1 is worth
debugging yet.

In [ ]:
RAW = f"https://raw.githubusercontent.com/SeonghwanSeo/PharmacoNet/{COMMIT}/examples/"
for name in ("6OIM_protein.pdb", "6OIM_D_MOV.pdb"):
    if not Path(name).exists():
        urllib.request.urlretrieve(RAW + name, name)

print({name: Path(name).stat().st_size for name in ("6OIM_protein.pdb", "6OIM_D_MOV.pdb")})

Build the model on CPU and run it.

`run()` takes the protein and a reference ligand whose centre of mass defines the box it looks at
(64 x 64 x 64 voxels at 0.5 angstrom, so a 32 angstrom cube). Expect roughly a minute.

> **Note:** if `numba` misbehaves, pass `molvoxel_library="numpy"` instead. PharmacoNet falls back
> to numpy on its own if numba is missing, but not if numba is present and broken.

In [ ]:
import time

module = None
try:
    start = time.time()
    module = PharmacoNet(device="cpu", weight_path=str(WEIGHTS), verbose=False)
    reference = module.run("6OIM_protein.pdb", ref_ligand_path="6OIM_D_MOV.pdb")
    check("4. reference run (6OIM)", len(reference.nodes) > 0,
          f"{len(reference.nodes)} features in {time.time() - start:.0f} s")
except Exception as error:
    check("4. reference run (6OIM)", False, f"{type(error).__name__}: {error}")

## 5. Load the CpABC1 target

PharmacoNet needs two things: a protein PDB, and any file whose coordinates mark which pocket to
look at. We use `cpabc1_receptor.pdb` (protein only, 11435 heavy atoms, 1431 residues, chain A)
and `silymarin_ligand.sdf` (silybin, 35 heavy atoms, C25H22O10, in the same coordinate frame).

These are **not** in this repository: the CpABC1 model is unpublished participant work and this
repo is public. Upload them from `projects/blue/data/` in your local clone.

In [ ]:
NEEDED = ["cpabc1_receptor.pdb", "silymarin_ligand.sdf"]
missing = [name for name in NEEDED if not Path(name).exists()]
if missing:
    from google.colab import files
    print("Upload from projects/blue/data/ in your clone:", ", ".join(missing))
    files.upload()

# Once the blue data is committed to main, use these two lines instead:
# !git clone -q --depth 1 https://github.com/ersilia-os/ub-cedd-projects-workshop /content/ws
# !cp /content/ws/projects/blue/data/{cpabc1_receptor.pdb,silymarin_ligand.sdf} .

check("5. target files present", all(Path(name).exists() for name in NEEDED),
      str({name: Path(name).stat().st_size for name in NEEDED if Path(name).exists()}))

Now the pre-flight that catches the one trap specific to this file.

PharmacoNet's pocket extraction keeps only residues whose name is in a hard-coded list of the 20
standard amino acids, and drops everything else **without a warning**. Our receptor was written by
Maestro, and Maestro writes `HIE`, `HID` or `HIP` for histidine and `CYX` for a disulphide-bonded
cysteine. If any of those are present they would be silently deleted, and PharmacoNet would model
the hole they left behind.

So check the residue names before running, not after.

In [ ]:
STANDARD = set("ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL".split())

atoms = [line for line in open("cpabc1_receptor.pdb") if line.startswith(("ATOM", "HETATM"))]
names = {line[17:20].strip() for line in atoms}
elements = sorted({line[76:78].strip() for line in atoms})
odd = names - STANDARD

print(f"{len(atoms)} atoms | elements {elements} | {len(names)} residue names")
check("5b. residue names standard", not odd, f"non-standard: {sorted(odd)}" if odd else "all standard")

## 6. Run PharmacoNet on CpABC1

The same two lines as section 4, with our files.

> **Note:** CpABC1 is a roughly 1431-residue membrane ABC transporter. PharmacoNet was trained on
> CrossDocked and Binding MOAD, which are soluble, globular, drug-like pockets. This target is out
> of distribution in three ways at once: it is a transporter rather than an enzyme, its site sits
> in a transmembrane cavity, and the pocket it extracts will pull in lipid-facing helices the
> model has never seen. A result here is a feasibility signal, not a validated pharmacophore.

In [ ]:
model = None
try:
    start = time.time()
    model = module.run("cpabc1_receptor.pdb", ref_ligand_path="silymarin_ligand.sdf")
    model.save("cpabc1_silybin.json")
    check("6. CpABC1 run", len(model.nodes) > 0,
          f"{len(model.nodes)} features in {time.time() - start:.0f} s")
except Exception as error:
    check("6. CpABC1 run", False, f"{type(error).__name__}: {error}")

## 7. Read the pharmacophore out

Each node is one pharmacophore feature. `type` is the feature a ligand would need; `center` is
where that ligand atom should sit; `hotspot_position` is the protein atom that would make the
interaction; `score` is the model's confidence.

That score is a **percentile, not a probability**: 0.9 means this hotspot scored higher than 90
per cent of the hotspots PharmacoNet saw during training.

In [ ]:
import pandas as pd

features = pd.DataFrame([{"type": node.type, "interaction": node.interaction_type,
                          "score": round(float(node.score), 3), "radius": round(float(node.radius), 2),
                          "cx": node.center[0], "cy": node.center[1], "cz": node.center[2]}
                         for node in model.nodes])

print(features["type"].value_counts().to_dict())
features.sort_values("score", ascending=False).head(15)

## 8. Do the features land on silybin?

This is what turns a smoke test into evidence.

PharmacoNet never sees silybin's atoms. It only uses the ligand file to pick the centre of the box.
So if the features it finds independently coincide with where silybin actually sits, the
pharmacophore is picking up real chemistry in the pocket rather than noise.

First the ligand side: how far is each feature's centre from the nearest silybin heavy atom?

In [ ]:
import numpy as np
from rdkit import Chem

ligand = Chem.MolFromMolFile("silymarin_ligand.sdf")
positions = ligand.GetConformer().GetPositions()
symbols = [atom.GetSymbol() for atom in ligand.GetAtoms()]

rows = []
for node in model.nodes:
    distances = np.linalg.norm(positions - np.asarray(node.center), axis=1)
    nearest = int(distances.argmin())
    rows.append({"type": node.type, "score": round(float(node.score), 3),
                 "nearest_atom": f"{symbols[nearest]}{nearest + 1}",
                 "distance": round(float(distances[nearest]), 2)})

on_ligand = pd.DataFrame(rows).sort_values("distance").reset_index(drop=True)
close = int((on_ligand.distance <= 3.0).sum())
check("8a. features on silybin", close >= 3, f"{close}/{len(on_ligand)} within 3.0 A")
on_ligand.head(20)

Now the protein side. Each feature's `hotspot_position` is a real protein atom, so we can
name the residue it belongs to and compare against the contacts measured directly from the
complex: **Ser862 (2.73 A), Asn858 (2.77), Ser888 (2.77), Thr353 (2.80), Tyr961 (2.87),
Gln1101 (2.90), Arg965 (3.05)**.

Be clear about the bar. PharmacoNet models the whole pocket, so it will legitimately find hotspots
silybin never touches - and that is the interesting part, because those are exactly where a better
molecule could gain affinity. So the test is "does it recover some of the contacts we know are
real", not "does it recover only those".

In [ ]:
CONTACTS = {"SER862", "ASN858", "SER888", "THR353", "TYR961", "GLN1101", "ARG965"}

protein = [line for line in open("cpabc1_receptor.pdb") if line.startswith("ATOM")]
coords = np.array([[float(line[30:38]), float(line[38:46]), float(line[46:54])] for line in protein])
residues = [f"{line[17:20].strip()}{int(line[22:26])}" for line in protein]

rows = []
for node in model.nodes:
    distances = np.linalg.norm(coords - np.asarray(node.hotspot_position), axis=1)
    nearest = int(distances.argmin())
    rows.append({"type": node.type, "residue": residues[nearest],
                 "distance": round(float(distances[nearest]), 2),
                 "known_contact": residues[nearest] in CONTACTS})

on_protein = pd.DataFrame(rows)
found = sorted({r for r in on_protein.residue if r in CONTACTS})
check("8b. known contacts recovered", len(found) >= 3, f"{len(found)}/7: {found}")
on_protein.sort_values("distance").head(20)

One last look, and a deliberate non-test.

Silybin is C25H22O10: ten oxygens, two aromatic systems, and **no ionisable group at pH 7**. So
expect hydrogen-bond donors and acceptors on the hydroxyls and the carbonyl, aromatic features on
the ring centroids, and hydrophobic ones on the methyl and the benzodioxane carbons.

If `Cation` or `Anion` features turn up, that is not a bug. It means the pocket offers charged
interactions that silybin does not exploit - which is an opening for a better molecule, not a
problem. That is why this is a table to read rather than a pass or fail.

In [ ]:
summary = (on_ligand.groupby("type")
           .agg(count=("type", "size"), best_score=("score", "max"), closest=("distance", "min"))
           .sort_values("count", ascending=False))

charged = int(summary.reindex(["Cation", "Anion"])["count"].sum(skipna=True) or 0)
print(f"{charged} charged features - silybin has no ionisable group, so these are pocket chemistry it leaves unused")
summary

## 9. Look at it in the pocket

PharmacoNet writes a PyMOL session file, but `pymol-open-source` is conda-only and installing conda
into Colab is exactly what makes the PharmacoForge notebook slow. We draw the same picture with
py3Dmol instead, using the authors' own colours so it matches the figures in their paper.

Seeing the spheres sit around silybin, rather than drifting off into the lipid or the solvent, is
the check that nothing went wrong geometrically.

In [ ]:
import py3Dmol

COLOURS = {"Hydrophobic": "orange", "Aromatic": "purple", "Cation": "blue", "Anion": "red",
           "HBond_acceptor": "magenta", "HBond_donor": "cyan", "Halogen": "yellow"}

view = py3Dmol.view(width="100%", height=500)
view.addModel(open("cpabc1_receptor.pdb").read(), "pdb")
view.setStyle({"cartoon": {"color": "lightgrey", "opacity": 0.6}})
view.addModel(open("silymarin_ligand.sdf").read(), "sdf")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})
for node in model.nodes:
    view.addSphere({"center": dict(zip("xyz", map(float, node.center))),
                    "radius": float(node.radius) / 2,
                    "color": COLOURS.get(node.type, "white"), "wireframe": True})
view.zoomTo({"model": 1})
view.show()

## 10. Record the environment

`CLAUDE.md` promises a `colab-environment.txt` recording Colab's Python version and package
versions, and it was never created. Print the lines for it here so they can be pasted straight in.

In [ ]:
import datetime

PACKAGES = ["pharmaconet", "torch", "numpy", "numba", "rdkit", "molvoxel", "omegaconf",
            "gdown", "biopython", "openbabel-wheel", "py3Dmol", "pandas"]

lines = [f"# colab-environment.txt - {datetime.date.today()} - image {os.environ.get('COLAB_RELEASE_TAG', '?')}",
         f"python=={platform.python_version()}"]
for package in PACKAGES:
    try:
        lines.append(f"{package}=={meta.version(package)}")
    except meta.PackageNotFoundError:
        lines.append(f"# {package}: not installed")

print("\n".join(lines))

## 11. The verdict

Every section above recorded a pass or a fail, and the risky ones caught their own exceptions so
this cell is always reached. Copy the block it prints back into the chat.

In [ ]:
print(f"PHARMACONET SMOKE TEST | {datetime.date.today()} | "
      f"image {os.environ.get('COLAB_RELEASE_TAG', '?')} | python {platform.python_version()}")
print("-" * 72)
for step, (status, detail) in RESULT.items():
    print(f"{status:4}  {step:28}  {detail}")
print("-" * 72)
print(f"{sum(status == 'PASS' for status, _ in RESULT.values())}/{len(RESULT)} passed")

## Summary

- If sections 4 and 6 passed, PharmacoNet installs and runs in Colab from a plain pip install, with
  no conda and no kernel restart, and it produces a pharmacophore for CpABC1.
- If section 8 recovered several of silybin's known contacts, the pharmacophore is picking up real
  pocket chemistry rather than noise.
- Copy the block from section 11 back into the chat, along with the versions from section 10.

**If something failed, read it in this order:**

| Failing step | Almost certainly |
|---|---|
| 1 | The runtime version is not 2026.07. Nothing else is worth reading. |
| 2 | pip could not resolve, usually `numba` against Colab's `numpy`. Try `molvoxel_library="numpy"`. |
| 2b | `obabel` is missing from PATH. Mostly harmless, but worth knowing. |
| 3 | The Google Drive quota. Retry tomorrow, or mirror the file to this repo's releases. |
| 4 but not 6 | The install is fine and the problem is our input. Go back to 5b. |
| 5b | A non-standard residue name, which the pocket extraction deletes without warning. |
| 6 only | The target itself: size, memory, or a pocket too far out of distribution. |

**Known rough edges, all upstream:**

- `modeling.py`, the documented command-line entry point, imports PyMOL at module scope, so it
  cannot run without conda. This notebook uses the Python API instead, which is what `modeling.py`
  itself calls.
- No PyPI release, and one misleadingly named tag, so we pin a commit.
- The weights come from one Google Drive link with no mirror
  ([issue #12](https://github.com/SeonghwanSeo/PharmacoNet/issues/12)).
- The pocket extraction shells out to `obabel` and ignores both its error output and its exit code.

**Next:** if this passes, build `blue_pharmacophore_modelling.ipynb` on the same complex, and
decide whether to commit the CpABC1 files so it can read `data/` instead of asking for an upload.